# Model 1 v1: Road Closure Probability Prediction

Outputs generated by this notebook:

- `outputs/model1_v1/model1_v1_calibrated_xgb_model.pkl`
- `outputs/model1_v1/model1_v1_feature_importance.csv`
- `outputs/model1_v1/model1_v1_leakage_audit.csv`
- `outputs/model1_v1/model1_v1_test_predictions.csv`
- `outputs/model1_v1/model1_v1_duration_band_with_road_closure_features.csv`

## 1. Imports

In [13]:
import json
import warnings
from pathlib import Path

from joblib.externals import cloudpickle
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

warnings.filterwarnings('ignore')


## 2. Configuration and Paths

In [14]:
MODEL_VERSION = 'model1_v1'
ROAD_CLOSURE_TARGET = 'target_road_closure'
DURATION_TARGET = 'duration_band'
ROAD_CLOSURE_INPUT = 'model_ready_road_closure.csv'
DURATION_INPUT = 'model_ready_duration_band.csv'
DECISION_THRESHOLD = 0.5
RANDOM_STATE = 42
OPTUNA_N_TRIALS = 30
XGB_DEFAULT_BOOST_ROUNDS = 1000
XGB_TUNING_BOOST_ROUNDS = 500
XGB_FINAL_BOOST_ROUNDS = 1000
EARLY_STOPPING_ROUNDS = 50
PROJECT_ROOT_OVERRIDE = None


def resolve_project_root():
    if PROJECT_ROOT_OVERRIDE is not None:
        project_root = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if not (project_root / 'outputs' / ROAD_CLOSURE_INPUT).exists():
            raise FileNotFoundError(f'Input file not found under PROJECT_ROOT_OVERRIDE: {project_root}')
        return project_root

    cwd = Path.cwd().resolve()

    if IN_COLAB:
        drive.mount('/content/drive')
        candidates = [
            Path('/content/drive/MyDrive/flipkart_gridlock/Road_closure_classifier/GridLock_Phase2-main'),
            Path('/content/drive/MyDrive/Road_closure_classifier/GridLock_Phase2-main'),
            cwd,
        ]
    else:
        candidates = []
        for root in [cwd, *cwd.parents]:
            candidates.append(root)
            candidates.append(root / 'Phase 2' / 'theme 2')

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / 'outputs' / ROAD_CLOSURE_INPUT).exists():
            return candidate

    raise FileNotFoundError(
        'Could not find project root. Set PROJECT_ROOT_OVERRIDE to the folder containing the outputs directory.'
    )


PROJECT_ROOT = resolve_project_root()
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
MODEL_OUTPUT_DIR = OUTPUTS_DIR / MODEL_VERSION
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ROAD_CLOSURE_DATA_PATH = OUTPUTS_DIR / ROAD_CLOSURE_INPUT
DURATION_DATA_PATH = OUTPUTS_DIR / DURATION_INPUT
MODEL_ARTIFACT_PATH = MODEL_OUTPUT_DIR / f'{MODEL_VERSION}_calibrated_xgb_model.pkl'
FEATURE_IMPORTANCE_PATH = MODEL_OUTPUT_DIR / f'{MODEL_VERSION}_feature_importance.csv'
LEAKAGE_AUDIT_PATH = MODEL_OUTPUT_DIR / f'{MODEL_VERSION}_leakage_audit.csv'
TEST_PREDICTIONS_PATH = MODEL_OUTPUT_DIR / f'{MODEL_VERSION}_test_predictions.csv'
MODEL2_HANDOFF_PATH = MODEL_OUTPUT_DIR / f'{MODEL_VERSION}_duration_band_with_road_closure_features.csv'

print(f'Project root: {PROJECT_ROOT}')
print(f'Model output directory: {MODEL_OUTPUT_DIR}')


Project root: D:\Python\Gridlock\Phase 2\theme 2
Model output directory: D:\Python\Gridlock\Phase 2\theme 2\outputs\model1_v1


## 3. Load Input Data

In [15]:
road_closure_df = pd.read_csv(ROAD_CLOSURE_DATA_PATH)
duration_df = pd.read_csv(DURATION_DATA_PATH)

required_targets = {
    ROAD_CLOSURE_TARGET: road_closure_df,
    DURATION_TARGET: duration_df,
}
for target_name, frame in required_targets.items():
    if target_name not in frame.columns:
        raise ValueError(f'Missing target column: {target_name}')

print('Road-closure training data:', road_closure_df.shape)
print('Duration-band handoff data:', duration_df.shape)


Road-closure training data: (8173, 516)
Duration-band handoff data: (2764, 516)


## 4. Leakage Audit

This section identifies features that should not be used by Model 1 because they either encode endpoint/route-completion information or behave like near-perfect one-column predictors of `target_road_closure`. The detected columns are saved to a leakage audit CSV and excluded from training, inference features, artifacts, and the Model 2 handoff input columns.

In [16]:
raw_feature_cols = [col for col in road_closure_df.columns if col != ROAD_CLOSURE_TARGET]
road_closure_target = road_closure_df[ROAD_CLOSURE_TARGET].copy().astype(int)

LEAKAGE_NAME_PATTERNS = [
    'end_location',
    'end_point',
    'route_distance',
    'route_span',
]
LEAKAGE_BINARY_RULE_ACCURACY_THRESHOLD = 0.995
LEAKAGE_AUC_STRENGTH_THRESHOLD = 0.995
LEAKAGE_CORRELATION_THRESHOLD = 0.98

leakage_rows = []
for col in raw_feature_cols:
    lowered = col.lower()
    name_flag = any(pattern in lowered for pattern in LEAKAGE_NAME_PATTERNS)
    series = pd.to_numeric(road_closure_df[col], errors='coerce').replace([np.inf, -np.inf], np.nan)
    valid = series.notna() & road_closure_target.notna()
    auc_strength = np.nan
    correlation = np.nan
    binary_rule_accuracy = np.nan
    binary_rule_direction = ''

    if valid.sum() > 0 and road_closure_target.loc[valid].nunique() == 2:
        y_valid = road_closure_target.loc[valid]
        x_valid = series.loc[valid]
        if x_valid.nunique(dropna=True) > 1:
            correlation = x_valid.corr(y_valid)
            auc_value = roc_auc_score(y_valid, x_valid)
            auc_strength = max(auc_value, 1 - auc_value)

        unique_values = set(x_valid.dropna().unique())
        if unique_values.issubset({0, 1, 0.0, 1.0}):
            direct_pred = x_valid.astype(int)
            direct_accuracy = (direct_pred == y_valid).mean()
            inverted_accuracy = ((1 - direct_pred) == y_valid).mean()
            if direct_accuracy >= inverted_accuracy:
                binary_rule_accuracy = direct_accuracy
                binary_rule_direction = 'direct'
            else:
                binary_rule_accuracy = inverted_accuracy
                binary_rule_direction = 'inverted'

    auto_flag = (
        (pd.notna(binary_rule_accuracy) and binary_rule_accuracy >= LEAKAGE_BINARY_RULE_ACCURACY_THRESHOLD)
        or (pd.notna(auc_strength) and auc_strength >= LEAKAGE_AUC_STRENGTH_THRESHOLD)
        or (pd.notna(correlation) and abs(correlation) >= LEAKAGE_CORRELATION_THRESHOLD)
    )

    if name_flag or auto_flag:
        reasons = []
        if name_flag:
            reasons.append('name_pattern')
        if pd.notna(binary_rule_accuracy) and binary_rule_accuracy >= LEAKAGE_BINARY_RULE_ACCURACY_THRESHOLD:
            reasons.append('near_perfect_binary_rule')
        if pd.notna(auc_strength) and auc_strength >= LEAKAGE_AUC_STRENGTH_THRESHOLD:
            reasons.append('near_perfect_auc')
        if pd.notna(correlation) and abs(correlation) >= LEAKAGE_CORRELATION_THRESHOLD:
            reasons.append('near_perfect_correlation')

        leakage_rows.append(
            {
                'feature': col,
                'reasons': '|'.join(reasons),
                'binary_rule_accuracy': binary_rule_accuracy,
                'binary_rule_direction': binary_rule_direction,
                'auc_strength': auc_strength,
                'correlation': correlation,
                'non_null_rows': int(valid.sum()),
                'unique_values': int(series.nunique(dropna=True)),
            }
        )

leakage_audit_df = pd.DataFrame(
    leakage_rows,
    columns=[
        'feature',
        'reasons',
        'binary_rule_accuracy',
        'binary_rule_direction',
        'auc_strength',
        'correlation',
        'non_null_rows',
        'unique_values',
    ],
).sort_values(['binary_rule_accuracy', 'auc_strength', 'correlation'], ascending=False, na_position='last')

LEAKAGE_FEATURES = sorted(leakage_audit_df['feature'].tolist())
LEAKAGE_REVIEW_FEATURES = sorted(
    col for col in raw_feature_cols
    if col.startswith('past_closure_rate_') and col not in LEAKAGE_FEATURES
)

leakage_audit_df.to_csv(LEAKAGE_AUDIT_PATH, index=False)

print(f'Saved leakage audit CSV: {LEAKAGE_AUDIT_PATH}')
print(f'Detected leakage features to exclude: {len(LEAKAGE_FEATURES)}')
for feature in LEAKAGE_FEATURES:
    print(f'  - {feature}')
print(f'Past target-rate features kept for review: {len(LEAKAGE_REVIEW_FEATURES)}')


Saved leakage audit CSV: D:\Python\Gridlock\Phase 2\theme 2\outputs\model1_v1\model1_v1_leakage_audit.csv
Detected leakage features to exclude: 4
  - end_point_missing_or_zero
  - has_end_location
  - has_route_span
  - route_distance_km
Past target-rate features kept for review: 7


## 5. Prepare Features and Chronological Split

In [17]:
feature_cols = [col for col in raw_feature_cols if col not in LEAKAGE_FEATURES]

duration_feature_cols = [col for col in duration_df.columns if col != DURATION_TARGET and col not in LEAKAGE_FEATURES]
if feature_cols != duration_feature_cols:
    missing_in_duration = sorted(set(feature_cols) - set(duration_feature_cols))
    extra_in_duration = sorted(set(duration_feature_cols) - set(feature_cols))
    raise ValueError(
        'Road-closure and duration-band feature schemas do not match. '
        f'Missing in duration: {missing_in_duration[:20]}; extra in duration: {extra_in_duration[:20]}'
    )

print(f'Excluded leakage features: {len(LEAKAGE_FEATURES)}')
for feature in LEAKAGE_FEATURES:
    print(f'  - {feature}')
print(f'Total features available after leakage removal: {len(feature_cols)}')
print('First 20 features:')
for i, col in enumerate(feature_cols[:20], 1):
    print(f'  {i}. {col}')
print(f'  ... and {max(len(feature_cols) - 20, 0)} more features')

X_all = road_closure_df[feature_cols].copy()
y_all = road_closure_target.copy()

print(f'Feature matrix: {X_all.shape}')
print(f'Target vector: {y_all.shape}')

missing = X_all.isnull().sum()
if missing.sum() > 0:
    print(f'Total missing values: {missing.sum():,}')
    print('Columns with missing values:')
    for col in missing[missing > 0].index:
        print(f'  {col}: {missing[col]} ({missing[col] / len(X_all) * 100:.2f}%)')

    for col in X_all.columns:
        if X_all[col].isnull().sum() == 0:
            continue
        if pd.api.types.is_numeric_dtype(X_all[col]):
            fill_value = X_all[col].median()
        else:
            mode_values = X_all[col].mode(dropna=True)
            fill_value = mode_values.iloc[0] if not mode_values.empty else 0
        X_all[col] = X_all[col].fillna(fill_value)
    print('Missing values imputed')
else:
    print('No missing values found')

numeric_columns = X_all.select_dtypes(include=[np.number]).columns
inf_check = np.isinf(X_all[numeric_columns]).sum() if len(numeric_columns) else pd.Series(dtype=int)
if inf_check.sum() > 0:
    print(f'Infinite values found: {int(inf_check.sum())}')
    X_all = X_all.replace([np.inf, -np.inf], np.nan)
    X_all = X_all.fillna(X_all.median(numeric_only=True))
    print('Infinite values handled')
else:
    print('No infinite values found')

X_all = X_all.apply(pd.to_numeric, errors='raise')
feature_medians = X_all.median(numeric_only=True).reindex(feature_cols).fillna(0)

n_rows = len(X_all)
train_end = int(0.70 * n_rows)
val_end = int(0.85 * n_rows)

X_train = X_all.iloc[:train_end].copy()
y_train = y_all.iloc[:train_end].copy()
X_val = X_all.iloc[train_end:val_end].copy()
y_val = y_all.iloc[train_end:val_end].copy()
X_test = X_all.iloc[val_end:].copy()
y_test = y_all.iloc[val_end:].copy()

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

split_summary = pd.DataFrame(
    {
        'split': ['train', 'validation', 'test'],
        'rows': [len(X_train), len(X_val), len(X_test)],
        'closure_rate': [y_train.mean(), y_val.mean(), y_test.mean()],
    }
)
print(split_summary)
print(f'Scale positive weight: {scale_pos_weight:.4f}')

dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)


Excluded leakage features: 4
  - end_point_missing_or_zero
  - has_end_location
  - has_route_span
  - route_distance_km
Total features available after leakage removal: 511
First 20 features:
  1. latitude
  2. longitude
  3. valid_start_coordinate
  4. has_start_location
  5. distance_to_city_center_km
  6. start_hour
  7. start_dayofweek
  8. start_weekofyear
  9. is_weekend
  10. is_peak_hour
  11. is_night
  12. hour_sin
  13. hour_cos
  14. day_sin
  15. day_cos
  16. report_lag_minutes_clipped
  17. report_lag_hours_clipped
  18. report_lag_is_negative
  19. reporting_delay_minutes
  20. text_length
  ... and 491 more features
Feature matrix: (8173, 511)
Target vector: (8173,)
No missing values found
No infinite values found
        split  rows  closure_rate
0       train  5721      0.065373
1  validation  1226      0.103589
2        test  1226      0.142741
Scale positive weight: 14.2968


## 6. Train Optuna-Tuned Calibrated XGBoost Model

In [18]:
def predict_booster_probability(booster, frame):
    dmatrix = xgb.DMatrix(frame)
    best_iteration = getattr(booster, 'best_iteration', None)
    if best_iteration is None or best_iteration <= 0:
        return booster.predict(dmatrix)
    return booster.predict(dmatrix, iteration_range=(0, best_iteration))


class XGBBoosterClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, booster=None, feature_columns=None):
        self.booster = booster
        self.feature_columns = feature_columns
        self._estimator_type = 'classifier'

    def fit(self, X, y=None):
        self.classes_ = np.array([0, 1])
        self.n_features_in_ = len(self.feature_columns) if self.feature_columns is not None else X.shape[1]
        self.is_fitted_ = True
        return self

    def _prepare_frame(self, X):
        if isinstance(X, pd.DataFrame):
            prepared = X.copy()
        else:
            prepared = pd.DataFrame(X, columns=self.feature_columns)
        if self.feature_columns is not None:
            prepared = prepared.loc[:, self.feature_columns]
        return prepared

    def predict_proba(self, X):
        prepared = self._prepare_frame(X)
        probability = predict_booster_probability(self.booster, prepared)
        return np.vstack([1 - probability, probability]).T

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= DECISION_THRESHOLD).astype(int)


def make_prefit_calibrator(fitted_estimator, method='sigmoid'):
    return CalibratedClassifierCV(FrozenEstimator(fitted_estimator), method=method)


xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': ['auc', 'logloss'],
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos_weight,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
}

print('Training XGBoost (default params)...')
default_xgb_model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=XGB_DEFAULT_BOOST_ROUNDS,
    evals=[(dtrain, 'train'), (dval, 'valid')],
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    verbose_eval=100,
)

xgb_val_proba = predict_booster_probability(default_xgb_model, X_val)
default_xgb_metrics = {
    'validation_roc_auc': float(roc_auc_score(y_val, xgb_val_proba)),
    'validation_pr_auc': float(average_precision_score(y_val, xgb_val_proba)),
    'validation_brier_score': float(brier_score_loss(y_val, xgb_val_proba)),
    'best_iteration': int(default_xgb_model.best_iteration),
}
print('Default XGBoost validation metrics:')
print(json.dumps(default_xgb_metrics, indent=2))


def objective_xgb(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'aucpr',
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'scale_pos_weight': scale_pos_weight,
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
    }

    booster = xgb.train(
        params,
        dtrain,
        num_boost_round=XGB_TUNING_BOOST_ROUNDS,
        evals=[(dval, 'val')],
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        verbose_eval=False,
    )

    preds = predict_booster_probability(booster, X_val)
    return average_precision_score(y_val, preds)


print(f'Starting Optuna tuning for XGBoost ({OPTUNA_N_TRIALS} trials)...')
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=OPTUNA_N_TRIALS, show_progress_bar=True)
print(f'Best XGBoost PR-AUC: {study_xgb.best_value:.4f}')
print('Best XGBoost hyperparameters:')
print(json.dumps(study_xgb.best_params, indent=2))

best_xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',
    'scale_pos_weight': scale_pos_weight,
    **study_xgb.best_params,
}

print('Training final tuned XGBoost model...')
final_xgb_model = xgb.train(
    best_xgb_params,
    dtrain,
    num_boost_round=XGB_FINAL_BOOST_ROUNDS,
    evals=[(dtrain, 'train'), (dval, 'valid')],
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    verbose_eval=100,
)

xgb_wrapper = XGBBoosterClassifier(final_xgb_model, feature_cols).fit(X_train, y_train)
calibrated_xgb = make_prefit_calibrator(xgb_wrapper, method='sigmoid')
calibrated_xgb.fit(X_val, y_val)

raw_probs = xgb_wrapper.predict_proba(X_test)[:10, 1]
cal_probs = calibrated_xgb.predict_proba(X_test)[:10, 1]
print('Sample of Raw vs Calibrated Probabilities (XGBoost):')
for raw_probability, calibrated_probability in zip(raw_probs, cal_probs):
    print(f'Raw: {raw_probability:.4f} -> Calibrated: {calibrated_probability:.4f}')


Training XGBoost (default params)...
[0]	train-auc:0.98470	train-logloss:0.64643	valid-auc:0.97501	valid-logloss:0.64709
[100]	train-auc:0.99999	train-logloss:0.01473	valid-auc:0.99933	valid-logloss:0.02891
[200]	train-auc:1.00000	train-logloss:0.00303	valid-auc:0.99944	valid-logloss:0.02023
[300]	train-auc:1.00000	train-logloss:0.00149	valid-auc:0.99942	valid-logloss:0.01989
[357]	train-auc:1.00000	train-logloss:0.00117	valid-auc:0.99940	valid-logloss:0.02015


[I 2026-06-19 00:57:29,851] A new study created in memory with name: no-name-00be9d4f-cfe1-4ee4-b81c-b8d66973e1c4


Default XGBoost validation metrics:
{
  "validation_roc_auc": 0.9994483173679723,
  "validation_pr_auc": 0.9948556288989034,
  "validation_brier_score": 0.005234950687736273,
  "best_iteration": 307
}
Starting Optuna tuning for XGBoost (30 trials)...


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-06-19 00:57:40,645] Trial 0 finished with value: 0.994335022440745 and parameters: {'max_depth': 4, 'learning_rate': 0.015001211351984716, 'subsample': 0.9273348962787019, 'colsample_bytree': 0.9952546471091537, 'gamma': 0.00022592573971150703}. Best is trial 0 with value: 0.994335022440745.
[I 2026-06-19 00:57:46,936] Trial 1 finished with value: 0.992932458054901 and parameters: {'max_depth': 9, 'learning_rate': 0.04056496769599239, 'subsample': 0.8641005134268236, 'colsample_bytree': 0.934004318556749, 'gamma': 3.026904984649504e-08}. Best is trial 0 with value: 0.994335022440745.
[I 2026-06-19 00:57:51,366] Trial 2 finished with value: 0.9931210363666206 and parameters: {'max_depth': 6, 'learning_rate': 0.03355166194340501, 'subsample': 0.7563854228193606, 'colsample_bytree': 0.9611524185339247, 'gamma': 0.0049166019684665635}. Best is trial 0 with value: 0.994335022440745.
[I 2026-06-19 00:57:57,024] Trial 3 finished with value: 0.9935740389393032 and parameters: {'max_dep

## 7. Evaluate Model

In [19]:
val_probability = calibrated_xgb.predict_proba(X_val)[:, 1]
test_probability = calibrated_xgb.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= DECISION_THRESHOLD).astype(int)

metrics = {
    'model_version': MODEL_VERSION,
    'decision_threshold': DECISION_THRESHOLD,
    'training_logic': 'xgb.train + optuna + sigmoid calibration',
    'optuna_trials': OPTUNA_N_TRIALS,
    'default_xgb_validation': default_xgb_metrics,
    'optuna_best_validation_pr_auc': float(study_xgb.best_value),
    'best_xgb_params': best_xgb_params,
    'validation_roc_auc': float(roc_auc_score(y_val, val_probability)),
    'validation_pr_auc': float(average_precision_score(y_val, val_probability)),
    'validation_brier_score': float(brier_score_loss(y_val, val_probability)),
    'test_roc_auc': float(roc_auc_score(y_test, test_probability)),
    'test_pr_auc': float(average_precision_score(y_test, test_probability)),
    'test_brier_score': float(brier_score_loss(y_test, test_probability)),
    'test_precision': float(precision_score(y_test, test_prediction, zero_division=0)),
    'test_recall': float(recall_score(y_test, test_prediction, zero_division=0)),
    'test_confusion_matrix': confusion_matrix(y_test, test_prediction).tolist(),
}

test_predictions_df = pd.DataFrame(
    {
        'actual_label': y_test.values,
        'predicted_label': test_prediction,
        'prediction_probability': test_probability,
        'prediction_percentage': test_probability * 100,
    },
    index=y_test.index,
)
test_predictions_df.to_csv(TEST_PREDICTIONS_PATH, index=False)

print(json.dumps(metrics, indent=2))
print('\nClassification report:')
print(classification_report(y_test, test_prediction, target_names=['No Closure', 'Road Closure']))
print(f'\nSaved test predictions CSV: {TEST_PREDICTIONS_PATH}')


{
  "model_version": "model1_v1",
  "decision_threshold": 0.5,
  "training_logic": "xgb.train + optuna + sigmoid calibration",
  "optuna_trials": 30,
  "default_xgb_validation": {
    "validation_roc_auc": 0.9994483173679723,
    "validation_pr_auc": 0.9948556288989034,
    "validation_brier_score": 0.005234950687736273,
    "best_iteration": 307
  },
  "optuna_best_validation_pr_auc": 0.9959892584365249,
  "best_xgb_params": {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "scale_pos_weight": 14.296791443850267,
    "max_depth": 4,
    "learning_rate": 0.05353712097525424,
    "subsample": 0.8116555806722591,
    "colsample_bytree": 0.6533533649411021,
    "gamma": 5.8046988329444224e-05
  },
  "validation_roc_auc": 0.9994698114964928,
  "validation_pr_auc": 0.9952248230522585,
  "validation_brier_score": 0.00535796095986627,
  "test_roc_auc": 0.9977110235150197,
  "test_pr_auc": 0.9830511795922285,
  "test_brier_score": 0.009468207797188146,
  "test_precision": 0

## 8. Feature Importance

This section ranks the trained XGBoost features by importance. `gain` is the main ranking column because it measures how much a feature improves tree splits when used. The CSV is saved with gain, split count, and cover so the values can be reviewed later without rerunning the notebook.

In [20]:
importance_gain = final_xgb_model.get_score(importance_type='gain')
importance_weight = final_xgb_model.get_score(importance_type='weight')
importance_cover = final_xgb_model.get_score(importance_type='cover')

def score_for(score, feature, index):
    return score.get(feature, score.get(f'f{index}', 0.0))


feature_importance_df = pd.DataFrame({'feature': feature_cols})
feature_importance_df['gain'] = [score_for(importance_gain, feature, i) for i, feature in enumerate(feature_cols)]
feature_importance_df['weight'] = [score_for(importance_weight, feature, i) for i, feature in enumerate(feature_cols)]
feature_importance_df['cover'] = [score_for(importance_cover, feature, i) for i, feature in enumerate(feature_cols)]
feature_importance_df = feature_importance_df.sort_values('gain', ascending=False).reset_index(drop=True)

feature_importance_df.to_csv(FEATURE_IMPORTANCE_PATH, index=False)

print(f'Saved feature importance CSV: {FEATURE_IMPORTANCE_PATH}')
print(feature_importance_df.head(30).to_string(index=False))


Saved feature importance CSV: D:\Python\Gridlock\Phase 2\theme 2\outputs\model1_v1\model1_v1_feature_importance.csv
                                                   feature       gain  weight       cover
                     event_cause_grouped_vehicle_breakdown 159.444870     1.0  331.818787
                             event_cause_grouped_tree_fall 149.892242     1.0 1577.715332
                                               text_length 141.583755   497.0  270.709778
corridor_cause_interaction_grouped_orr east 2_construction  87.951996     1.0  434.180267
                               corridor_grouped_orr east 2  75.339066     2.0  186.585388
          zone_cause_interaction_grouped_unknown_pot_holes  74.295044     1.0  142.460983
                    junction_grouped_devasandra(k r puram)  62.154953     4.0  743.051331
                                   description_char_length  61.732613   368.0  148.206284
                                          has_kannada_text  47.356163     

## 9. Save Model Artifact

The model is already trained above. This cell saves the trained pipeline as a `.pkl` artifact so future model-ready input files can be scored without retraining.

To use this artifact later with the same model and feature schema:

```python
import joblib
import numpy as np
import pandas as pd

artifact = joblib.load('outputs/model1_v1/model1_v1_calibrated_xgb_model.pkl')
next_input = pd.read_csv('outputs/model_ready_road_closure.csv')

feature_cols = artifact['feature_cols']
feature_medians = artifact['feature_medians']
threshold = artifact['decision_threshold']
model = artifact['model']

X_next = next_input[feature_cols].copy()
X_next = X_next.replace([np.inf, -np.inf], np.nan)
X_next = X_next.apply(pd.to_numeric, errors='raise')
X_next = X_next.fillna(feature_medians)

probability = model.predict_proba(X_next)[:, 1]
scored_output = next_input.copy()
scored_output['road_closure_predicted_label'] = (probability >= threshold).astype(int)
scored_output['road_closure_probability'] = probability
scored_output['road_closure_percentage'] = probability * 100
```

The next input must be generated with the same feature-engineering pipeline so its columns match `feature_cols`.

In [21]:
model_artifact = {
    'model_version': MODEL_VERSION,
    'model': calibrated_xgb,
    'raw_booster': final_xgb_model,
    'feature_cols': feature_cols,
    'feature_medians': feature_medians,
    'decision_threshold': DECISION_THRESHOLD,
    'scale_pos_weight': scale_pos_weight,
    'best_xgb_params': best_xgb_params,
    'default_xgb_metrics': default_xgb_metrics,
    'metrics': metrics,
    'test_predictions': test_predictions_df,
    'feature_importance': feature_importance_df,
    'leakage_features': LEAKAGE_FEATURES,
    'leakage_audit': leakage_audit_df,
    'leakage_review_features': LEAKAGE_REVIEW_FEATURES,
}

with MODEL_ARTIFACT_PATH.open('wb') as artifact_file:
    cloudpickle.dump(model_artifact, artifact_file)
print(f'Saved model artifact: {MODEL_ARTIFACT_PATH}')


Saved model artifact: D:\Python\Gridlock\Phase 2\theme 2\outputs\model1_v1\model1_v1_calibrated_xgb_model.pkl


## 10. Generate Model 2 Handoff Dataset

In [22]:
def prepare_model1_features(frame, feature_columns, medians):
    prepared = frame[feature_columns].copy()
    prepared = prepared.replace([np.inf, -np.inf], np.nan)
    prepared = prepared.apply(pd.to_numeric, errors='raise')
    return prepared.fillna(medians)


X_duration_for_model1 = prepare_model1_features(duration_df, feature_cols, feature_medians)
road_closure_probability = calibrated_xgb.predict_proba(X_duration_for_model1)[:, 1]

model2_handoff_df = duration_df[feature_cols].copy()
model2_handoff_df['road_closure_predicted_label'] = (road_closure_probability >= DECISION_THRESHOLD).astype(int)
model2_handoff_df['road_closure_probability'] = road_closure_probability
model2_handoff_df['road_closure_percentage'] = road_closure_probability * 100
model2_handoff_df[DURATION_TARGET] = duration_df[DURATION_TARGET]

model2_handoff_df.to_csv(MODEL2_HANDOFF_PATH, index=False)

X_model2 = model2_handoff_df.drop(columns=[DURATION_TARGET])
y_model2 = model2_handoff_df[DURATION_TARGET]

print(f'Saved Model 2 handoff CSV: {MODEL2_HANDOFF_PATH}')
print('Model 2 feature matrix:', X_model2.shape)
print('Model 2 target vector:', y_model2.shape)
print('Added Model 1 features:', [
    'road_closure_predicted_label',
    'road_closure_probability',
    'road_closure_percentage',
])


Saved Model 2 handoff CSV: D:\Python\Gridlock\Phase 2\theme 2\outputs\model1_v1\model1_v1_duration_band_with_road_closure_features.csv
Model 2 feature matrix: (2764, 514)
Model 2 target vector: (2764,)
Added Model 1 features: ['road_closure_predicted_label', 'road_closure_probability', 'road_closure_percentage']


## 11. Inference Helper

`predict_road_closure_probability(event_features)` is used when the notebook session already has the trained model in memory and you want to score one or more new model-ready event rows immediately. It applies the same feature order and missing-value handling used during training, calls the calibrated XGBoost model, and returns the predicted closure label, probability, and percentage. For a later session or production-style run, load the saved `.pkl` artifact instead of relying on this in-memory helper.

In [23]:
def predict_road_closure_probability(event_features):
    event_frame = prepare_model1_features(event_features, feature_cols, feature_medians)
    probability = calibrated_xgb.predict_proba(event_frame)[:, 1]
    percentage = probability * 100
    predicted_label = (probability >= DECISION_THRESHOLD).astype(int)

    return pd.DataFrame(
        {
            'road_closure_predicted_label': predicted_label,
            'road_closure_probability': probability,
            'road_closure_percentage': percentage,
        },
        index=event_features.index,
    )
